In [ ]:
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import StandardScaler

from rlaopt.atoms import Box, ElasticNet, L1Norm, SumSquares
from rlaopt.expression import Variable
from rlaopt.linalg import NystromConfig
from rlaopt.solvers import ADMM, ADMMConfig, ADMMState, ADMMStoppingCriteria

In [ ]:
# torch.set_default_dtype(torch.float32)
torch.set_default_dtype(torch.float64)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"

### Load data

In [ ]:
def standardize_and_convert_to_torch(X, y):
    # Standardize features and labels
    X_std = StandardScaler().fit_transform(X)
    y_std = StandardScaler().fit_transform(y.reshape(-1, 1)).flatten()

    X_torch = torch.tensor(X_std)
    y_torch = torch.tensor(y_std)

    return X_torch, y_torch


def load_data(file_path: str):
    X, y = load_svmlight_file(file_path)
    X_dense = X.toarray()

    X_torch, y_torch = standardize_and_convert_to_torch(X_dense, y)
    return X_torch, y_torch

In [ ]:
def gaussian_rand_features(X, n_features, bandwidth):
    W = (
        1
        / bandwidth
        * torch.randn((X.shape[1], n_features), device=X.device)
        / (n_features**0.5)
    )
    b = 2 * torch.pi * torch.rand((n_features,), device=X.device)
    return torch.cos(X @ W + b) * (2 / n_features) ** 0.5


def relu_rand_features(X, n_features):
    W = torch.randn((X.shape[1], n_features), device=X.device) / (n_features**0.5)
    return torch.relu(X @ W)

In [ ]:
# Taken from PROMISE paper (with more random features)
def load_acsincome():
    X = pd.read_pickle("./data/acsincome_data.pkl")
    y = pd.read_pickle("./data/acsincome_target.pkl")
    X_torch, y_torch = standardize_and_convert_to_torch(X.to_numpy(), y.to_numpy())
    X_torch = gaussian_rand_features(X_torch, 3000, bandwidth=1.0)
    return X_torch, y_torch


# Taken from GeNIOS paper
def load_e2006():
    X_torch, y_torch = load_data("./data/E2006.train.bz2")
    return X_torch, y_torch


# Taken from PROMISE paper
def load_realsim():
    X_torch, y_torch = load_data("./data/real-sim.bz2")
    return X_torch, y_torch


# Taken from PROMISE paper
def load_yearpredictionmsd():
    X_torch, y_torch = load_data("./data/YearPredictionMSD.bz2")
    X_torch = relu_rand_features(X_torch, X_torch.shape[0] // 100)
    return X_torch, y_torch

In [ ]:
X_torch, y_torch = load_acsincome()

In [ ]:
X_torch.shape

### Create optimization problem

In [ ]:
def create_constrained_elastic_net(X_torch, y_torch):
    X_T_y = X_torch.T @ y_torch
    lambd = 0.1 * torch.linalg.norm(X_T_y, ord=float("inf")) / X_torch.shape[0]
    w = Variable((X_torch.shape[1],), name="w")
    b = Variable((1,), name="b")
    obj = SumSquares(X_torch @ w + b - y_torch) * (0.5 / X_torch.shape[0]) + ElasticNet(
        w, l1_scaling=lambd, l2_scaling=lambd
    )
    constraints = Box(w, lower=0.0, upper=1.0)
    return obj, constraints, w, b


def create_robust_regression(X_torch, y_torch):
    w = Variable((X_torch.shape[1],), name="w")
    b = Variable((1,), name="b")
    obj = L1Norm(X_torch @ w + b - y_torch) * (1.0 / X_torch.shape[0])
    return obj, None, w, b


def combine_obj_constraints(obj, constraints):
    if constraints is None:
        return obj
    return obj + constraints

In [ ]:
obj, constraints, w, b = create_constrained_elastic_net(X_torch, y_torch)
# obj, constraints, w, b = create_robust_regression(X_torch, y_torch)
loss = combine_obj_constraints(obj, constraints)
loss = loss.to(device)

### Create solver

In [ ]:
def create_admm_solver(loss, n_features: int):
    precond_config = NystromConfig(
        rank_init=min(50, n_features // 10),
        base_damping=0.0,
    )
    return ADMM(loss, config=ADMMConfig(rho=1e0, preconditioner_config=precond_config))

In [ ]:
solver = create_admm_solver(loss, X_torch.shape[1])
num_iters = 300

### Run solver

In [ ]:
@dataclass(kw_only=True, frozen=True)
class StepResult:
    primal_residual: float
    dual_residual: float
    rho: float
    iteration: int
    time_elapsed: float


def step_result_from_state(state: ADMMState, time_elapsed: float) -> StepResult:
    return StepResult(
        primal_residual=state.primal_residual_norm,
        dual_residual=state.dual_residual_norm,
        rho=state.rho,
        iteration=state.iter_,
        time_elapsed=time_elapsed,
    )


def print_step_result_with_frequency(step_result: StepResult, frequency: int = 20):
    if step_result.iteration % frequency == 0:
        print(
            f"Iter: {step_result.iteration:4d} | "
            f"Primal Residual: {step_result.primal_residual:.4e} | "
            f"Dual Residual: {step_result.dual_residual:.4e} | "
            f"Rho: {step_result.rho:.4e} | "
            f"Time Elapsed: {step_result.time_elapsed:.2f}s"
        )

In [ ]:
def run_solver(loss, solver, num_iters):
    step_results = []

    params = loss.variable_values
    state = solver.init_state(params)

    step_results.append(step_result_from_state(state, 0.0))
    print_step_result_with_frequency(step_results[-1])

    for _ in range(num_iters):
        start_time = time.time()
        params, state = solver.step(params, state)
        time_elapsed = time.time() - start_time

        step_results.append(step_result_from_state(state, time_elapsed))
        print_step_result_with_frequency(step_results[-1])

    return params, step_results

In [ ]:
# params, step_results = run_solver(loss, solver, num_iters)

In [ ]:
result = solver.solve(
    stopping_criteria=ADMMStoppingCriteria(
        max_iters=num_iters, eps_abs=1e-4, eps_rel=1e-4
    )
)

In [ ]:
result

### Visualize results

In [ ]:
def plot_residuals(step_results):
    primal_residuals = [sr.primal_residual for sr in step_results]
    dual_residuals = [sr.dual_residual for sr in step_results]
    time_elapsed = [sr.time_elapsed for sr in step_results]
    time_cumsum = torch.cumsum(torch.tensor(time_elapsed), dim=0).tolist()

    plt.figure(figsize=(10, 6))
    plt.semilogy(time_cumsum, primal_residuals, label="Primal Residual")
    plt.semilogy(time_cumsum, dual_residuals, label="Dual Residual")
    plt.xlabel("Time Elapsed (s)")
    plt.ylabel("Residual Norm")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_rhos(step_results):
    iterations = [sr.iteration for sr in step_results]
    rhos = [sr.rho for sr in step_results]

    plt.figure(figsize=(10, 6))
    plt.semilogy(iterations, rhos, label="Rho")
    plt.xlabel("Iteration")
    plt.ylabel("Rho Value")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# plot_residuals(step_results)

In [ ]:
# plot_rhos(step_results)